# 7.1 模型的保存与加载

本 notebook 详细介绍 PyTorch 模型的序列化与反序列化，包括保存/加载的两种方式、跨设备迁移以及训练断点续传

### 学习目标
- 理解 PyTorch 序列化机制（torch.save / torch.load）
- 掌握两种模型保存方式及其优缺点
- 学会使用 map_location 进行跨设备加载
- 实现完整的 Checkpoint 断点续传

In [ ]:
import torch
import torch.nn as nn
import os
import tempfile

print(f"PyTorch 版本: {torch.__version__}")

## 1. 序列化基础

PyTorch 使用 Python 的 `pickle` 模块进行序列化，提供了两个核心函数：

- `torch.save(obj, f)`: 将对象保存到文件
- `torch.load(f)`: 从文件加载对象

这两个函数可以保存任意 Python 对象，但最常用于保存模型和张量

In [ ]:
# 创建临时目录用于演示
save_dir = tempfile.mkdtemp()
print(f"保存目录: {save_dir}")

# 保存和加载张量
tensor_a = torch.randn(3, 4)
path = os.path.join(save_dir, "tensor.pt")
torch.save(tensor_a, path)

tensor_b = torch.load(path, weights_only=True)
print(f"原始张量:\n{tensor_a}")
print(f"\n加载张量:\n{tensor_b}")
print(f"\n是否相等: {torch.equal(tensor_a, tensor_b)}")

In [ ]:
# 也可以保存字典、列表等 Python 对象
data = {
    "weights": torch.randn(5, 3),
    "bias": torch.randn(3),
    "epoch": 10,
    "loss": 0.05,
}

path = os.path.join(save_dir, "data.pt")
torch.save(data, path)

loaded_data = torch.load(path, weights_only=True)
print(f"加载的字典: {loaded_data.keys()}")
print(f"epoch: {loaded_data['epoch']}")
print(f"loss: {loaded_data['loss']}")

## 2. 定义一个示例模型

为了演示保存和加载，我们先定义一个简单的神经网络

In [ ]:
class SimpleNet(nn.Module):
    """简单的三层神经网络"""

    def __init__(self, input_dim: int = 10, hidden_dim: int = 20, output_dim: int = 2):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x


# 创建模型并查看参数
model = SimpleNet()
print(model)
print(f"\n参数总数: {sum(p.numel() for p in model.parameters())}")

## 3. 方式一：保存整个模型（不推荐）

使用 `torch.save(model, path)` 可以保存整个模型对象

**为什么不推荐？**
- 依赖 pickle，保存的是模型类的引用路径（如 `__main__.SimpleNet`）
- 加载时必须能访问到相同的类定义
- 如果移动了代码文件或重命名了类，加载会失败
- 保存的文件更大
- 跨项目共享模型不方便

In [ ]:
# 方式一：保存整个模型
path_full = os.path.join(save_dir, "model_full.pth")
torch.save(model, path_full)
print(f"模型已保存到: {path_full}")
print(f"文件大小: {os.path.getsize(path_full)} 字节")

# 加载整个模型
# weights_only=False 是因为保存了整个模型对象
model_loaded = torch.load(path_full, weights_only=False)
print(f"\n加载的模型:\n{model_loaded}")

# 验证参数一致性
x_test = torch.randn(1, 10)
model.eval()
model_loaded.eval()
with torch.no_grad():
    out1 = model(x_test)
    out2 = model_loaded(x_test)
print(f"\n输出一致: {torch.equal(out1, out2)}")

## 4. 方式二：保存 state_dict（推荐）

`state_dict` 是一个有序字典，包含模型所有可学习参数的名称和值

**优点：**
- 只保存参数值，文件更小
- 不依赖具体的类定义路径
- 可以灵活地加载到不同结构的模型中
- 是 PyTorch 官方推荐的方式

In [ ]:
# 查看 state_dict 的内容
state_dict = model.state_dict()
print("state_dict 中的参数:")
for name, param in state_dict.items():
    print(f"  {name}: shape={param.shape}, dtype={param.dtype}")

In [ ]:
# 方式二：保存 state_dict
path_state = os.path.join(save_dir, "model_state.pth")
torch.save(model.state_dict(), path_state)
print(f"state_dict 已保存到: {path_state}")
print(f"文件大小: {os.path.getsize(path_state)} 字节")

# 加载 state_dict
# 步骤1: 先创建模型实例（需要知道模型结构）
model_new = SimpleNet()

# 步骤2: 加载 state_dict
state_dict = torch.load(path_state, weights_only=True)

# 步骤3: 将 state_dict 加载到模型中
model_new.load_state_dict(state_dict)
print("\nstate_dict 加载成功")

# 验证
model_new.eval()
with torch.no_grad():
    out3 = model_new(x_test)
print(f"输出一致: {torch.equal(out1, out3)}")

In [ ]:
# 对比两种方式的文件大小
size_full = os.path.getsize(path_full)
size_state = os.path.getsize(path_state)
print(f"保存整个模型: {size_full} 字节")
print(f"保存 state_dict: {size_state} 字节")
print(f"state_dict 节省了 {size_full - size_state} 字节 ({(1 - size_state/size_full)*100:.1f}%)")

## 5. load_state_dict 的 strict 参数

默认情况下 `strict=True`，要求 state_dict 中的 key 和模型完全匹配

- `strict=True`: key 必须完全匹配，否则报错
- `strict=False`: 只加载匹配的 key，忽略多余或缺失的 key

In [ ]:
# strict=False 的使用场景：部分加载
class ExtendedNet(nn.Module):
    """扩展版模型，比 SimpleNet 多一层"""

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 20)  # 与 SimpleNet 相同
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(20, 20)  # 与 SimpleNet 相同
        self.fc3 = nn.Linear(20, 2)   # 与 SimpleNet 相同
        self.fc4 = nn.Linear(2, 5)    # 新增层
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        x = self.fc4(x)
        return x


extended_model = ExtendedNet()
state_dict = torch.load(path_state, weights_only=True)

# strict=False: 只加载匹配的参数，新增的 fc4 保持随机初始化
result = extended_model.load_state_dict(state_dict, strict=False)
print(f"缺失的 key: {result.missing_keys}")
print(f"多余的 key: {result.unexpected_keys}")
print("\n说明: fc4 的参数没有在 state_dict 中，保持了随机初始化")

## 6. map_location：跨设备加载

当模型在 GPU 上训练、需要在 CPU 上加载（或反过来）时，使用 `map_location` 参数

| 场景 | map_location |
|------|-------------|
| GPU -> CPU | `torch.device('cpu')` |
| CPU -> GPU | `torch.device('cuda:0')` |
| GPU:0 -> GPU:1 | `torch.device('cuda:1')` |
| 自动选择 | `lambda storage, loc: storage` |

In [ ]:
# 跨设备加载示例

# 场景1: GPU 上保存的模型加载到 CPU
# （即使当前没有 GPU，代码也能正常运行）
state_dict_cpu = torch.load(
    path_state,
    map_location=torch.device('cpu'),
    weights_only=True,
)
print("GPU -> CPU 加载成功")

# 场景2: 检测设备并自动选择
def get_device() -> torch.device:
    """自动检测可用设备"""
    if torch.cuda.is_available():
        return torch.device('cuda:0')
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


device = get_device()
print(f"\n当前设备: {device}")

# 使用 map_location 加载到当前设备
state_dict_device = torch.load(
    path_state,
    map_location=device,
    weights_only=True,
)

model_device = SimpleNet()
model_device.load_state_dict(state_dict_device)
model_device.to(device)
print(f"模型已加载到 {device}")

In [ ]:
# 使用 lambda 函数进行更灵活的设备映射
# 这种方式会将所有张量加载到 CPU 上
state_dict_lambda = torch.load(
    path_state,
    map_location=lambda storage, loc: storage,
    weights_only=True,
)
print("使用 lambda 加载到 CPU 成功")

# 查看加载后参数的设备
for name, param in state_dict_lambda.items():
    print(f"  {name}: device={param.device}")

## 7. Checkpoint：断点续传

训练深度学习模型时，可能因为各种原因中断（断电、OOM 等）。Checkpoint 机制可以保存训练的完整状态，以便从断点恢复训练

**Checkpoint 应该包含：**
- 模型的 state_dict
- 优化器的 state_dict（包含动量等状态）
- 学习率调度器的 state_dict
- 当前 epoch 和 step
- 最佳指标值
- 其他需要恢复的状态

In [ ]:
import torch.optim as optim


def save_checkpoint(
    model: nn.Module,
    optimizer: optim.Optimizer,
    scheduler,
    epoch: int,
    best_loss: float,
    path: str,
) -> None:
    """保存训练检查点

    Args:
        model: 模型
        optimizer: 优化器
        scheduler: 学习率调度器
        epoch: 当前 epoch
        best_loss: 最佳损失值
        path: 保存路径
    """
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_loss': best_loss,
    }
    torch.save(checkpoint, path)
    print(f"Checkpoint 保存到: {path} (epoch={epoch}, best_loss={best_loss:.4f})")


def load_checkpoint(
    path: str,
    model: nn.Module,
    optimizer: optim.Optimizer,
    scheduler,
    device: torch.device,
) -> int:
    """加载训练检查点

    Args:
        path: 检查点路径
        model: 模型
        optimizer: 优化器
        scheduler: 学习率调度器
        device: 加载设备

    Returns:
        下一个 epoch 编号
    """
    checkpoint = torch.load(path, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    epoch = checkpoint['epoch']
    best_loss = checkpoint['best_loss']
    print(f"Checkpoint 加载成功: epoch={epoch}, best_loss={best_loss:.4f}")
    return epoch + 1  # 返回下一个 epoch


print("检查点保存/加载函数定义完成")

## 8. 完整示例：带断点续传的训练

In [ ]:
# 创建合成数据
torch.manual_seed(42)
X_train = torch.randn(200, 10)
y_train = (X_train[:, 0] + X_train[:, 1] > 0).long()  # 简单的二分类

X_val = torch.randn(50, 10)
y_val = (X_val[:, 0] + X_val[:, 1] > 0).long()

print(f"训练集: X={X_train.shape}, y={y_train.shape}")
print(f"验证集: X={X_val.shape}, y={y_val.shape}")
print(f"类别分布: {torch.bincount(y_train)}")

In [ ]:
def train_with_checkpoint(
    total_epochs: int = 10,
    resume_path: str = None,
):
    """带断点续传的训练函数

    Args:
        total_epochs: 总训练轮数
        resume_path: 恢复训练的检查点路径，None 表示从头开始
    """
    device = get_device()

    # 创建模型、优化器、调度器
    model = SimpleNet(input_dim=10, hidden_dim=20, output_dim=2).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    criterion = nn.CrossEntropyLoss()

    start_epoch = 0
    best_loss = float('inf')

    # 如果提供了检查点路径，从检查点恢复
    if resume_path and os.path.exists(resume_path):
        print("=" * 50)
        print("从检查点恢复训练...")
        checkpoint = torch.load(resume_path, map_location=device, weights_only=True)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_loss = checkpoint['best_loss']
        print(f"恢复自 epoch {checkpoint['epoch']}, best_loss={best_loss:.4f}")
        print("=" * 50)
    else:
        print("从头开始训练...")

    # 数据迁移到设备
    X_tr = X_train.to(device)
    y_tr = y_train.to(device)
    X_v = X_val.to(device)
    y_v = y_val.to(device)

    # 训练循环
    for epoch in range(start_epoch, total_epochs):
        # --- 训练阶段 ---
        model.train()
        optimizer.zero_grad()
        output = model(X_tr)
        train_loss = criterion(output, y_tr)
        train_loss.backward()
        optimizer.step()

        # --- 验证阶段 ---
        model.eval()
        with torch.no_grad():
            val_output = model(X_v)
            val_loss = criterion(val_output, y_v)
            val_acc = (val_output.argmax(dim=1) == y_v).float().mean()

        # 学习率调度
        scheduler.step()

        # 打印进度
        lr = optimizer.param_groups[0]['lr']
        print(
            f"Epoch [{epoch+1}/{total_epochs}] "
            f"train_loss={train_loss:.4f} "
            f"val_loss={val_loss:.4f} "
            f"val_acc={val_acc:.4f} "
            f"lr={lr:.6f}"
        )

        # 保存最佳模型
        if val_loss < best_loss:
            best_loss = val_loss.item()
            best_path = os.path.join(save_dir, "best_model.pth")
            torch.save(model.state_dict(), best_path)
            print(f"  -> 新的最佳模型已保存 (val_loss={best_loss:.4f})")

        # 每 3 个 epoch 保存一次检查点
        if (epoch + 1) % 3 == 0:
            ckpt_path = os.path.join(save_dir, "checkpoint_latest.pth")
            save_checkpoint(model, optimizer, scheduler, epoch, best_loss, ckpt_path)

    print(f"\n训练完成! 最佳 val_loss={best_loss:.4f}")
    return model

In [ ]:
# 第一阶段：训练前 6 个 epoch
print("===== 第一阶段训练 =====")
model_phase1 = train_with_checkpoint(total_epochs=6)

In [ ]:
# 第二阶段：从检查点恢复，继续训练到 10 个 epoch
print("\n===== 第二阶段训练（恢复）=====")
ckpt_path = os.path.join(save_dir, "checkpoint_latest.pth")
model_phase2 = train_with_checkpoint(
    total_epochs=10,
    resume_path=ckpt_path,
)

## 9. 总结

### 两种保存方式对比

| 特性 | 保存整个模型 | 保存 state_dict |
|------|------------|----------------|
| 函数 | `torch.save(model, path)` | `torch.save(model.state_dict(), path)` |
| 加载 | `torch.load(path)` | `model.load_state_dict(torch.load(path))` |
| 文件大小 | 较大 | 较小 |
| 灵活性 | 低（依赖类定义路径） | 高（只需要相同的网络结构） |
| 推荐 | 不推荐 | 推荐 |

### map_location 使用场景

| 场景 | 用法 |
|------|------|
| GPU 模型在 CPU 加载 | `map_location=torch.device('cpu')` |
| CPU 模型在 GPU 加载 | `map_location=torch.device('cuda:0')` |
| 自动选择设备 | 使用 `get_device()` 函数 |

### Checkpoint 最佳实践
1. 保存 model + optimizer + scheduler + epoch + best_metric
2. 定期保存检查点（每 N 个 epoch）
3. 始终保存最佳模型
4. 使用 `map_location` 处理跨设备加载

---

## 练习题

### 基础题
1. 创建一个简单模型，分别使用两种方式保存，对比文件大小
2. 保存一个模型的 state_dict，加载到结构不同的模型中（使用 `strict=False`），观察哪些参数被加载了

### 进阶题
3. 实现一个完整的断点续传训练：
   - 创建一个分类模型
   - 训练 5 个 epoch 后保存检查点
   - 模拟中断（停止训练）
   - 从检查点恢复并继续训练 5 个 epoch
   - 验证恢复后的损失是连续的（不会跳变）

### 挑战题
4. 实现一个 `ModelSaver` 类，支持：
   - 保存最近 N 个检查点（自动删除旧的）
   - 保存最佳模型（基于验证指标）
   - 自动管理保存路径

完成本章学习后，请尝试 [exercises.md](./exercises.md) 中的更多练习题来巩固知识。